In [13]:
import mediapipe as mp
print(mp.__version__)  # Should print 0.9.3

0.10.35


In [17]:
import urllib.request
import os

model_path = "hand_landmarker.task"

if not os.path.exists(model_path):
    print("Downloading model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        model_path
    )
    print("Done!")
else:
    print("Model already exists!")

print("Saved to:", os.path.abspath(model_path))  # Shows exact path

Done!
Saved to: C:\Users\CFRSI 11\Desktop\Computer vision project\hand_landmarker.task


In [21]:
import os
model_path = os.path.abspath("hand_landmarker.task")  # ← absolute path, no ambiguity

options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=model_path),  # ← use this
    ...
)

SyntaxError: positional argument follows keyword argument (2130687919.py, line 7)

In [23]:
import cv2
import mediapipe as mp

# ── Setup ──────────────────────────────────────────────
latest_result = None

def on_result(result, output_image, timestamp_ms):
    global latest_result
    latest_result = result

options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=on_result
)

CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),         # Thumb
    (0,5),(5,6),(6,7),(7,8),         # Index
    (0,9),(9,10),(10,11),(11,12),    # Middle
    (0,13),(13,14),(14,15),(15,16),  # Ring
    (0,17),(17,18),(18,19),(19,20),  # Pinky
    (5,9),(9,13),(13,17)             # Palm
]

# ── Main Loop ──────────────────────────────────────────
cap = cv2.VideoCapture(0)
timestamp = 0

with mp.tasks.vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        success, img = cap.read()
        if not success:
            break

        # Send frame to detector
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        landmarker.detect_async(mp_img, timestamp)
        timestamp += 1

        # Draw results
        if latest_result and latest_result.hand_landmarks:
            h, w, _ = img.shape
            for hand in latest_result.hand_landmarks:
                pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand]

                # Connections
                for a, b in CONNECTIONS:
                    cv2.line(img, pts[a], pts[b], (0, 255, 0), 2)

                # Landmarks
                for i, (cx, cy) in enumerate(pts):
                    cv2.circle(img, (cx, cy), 6, (255, 0, 255), -1)
                    cv2.putText(img, str(i), (cx+5, cy+5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255,255,255), 1)

        # FPS display
        cv2.putText(img, "Press Q to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.imshow("Hand Tracking", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [25]:
import cv2
import mediapipe as mp
import numpy as np
import math
import os
import screen_brightness_control as sbc
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

# ── Audio Setup ────────────────────────────────────────
devices = AudioUtilities.GetSpeakers()
interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
volume = cast(interface, POINTER(IAudioEndpointVolume))
vol_range = volume.GetVolumeRange()  # (-65.25, 0.0)
min_vol, max_vol = vol_range[0], vol_range[1]

# ── MediaPipe Setup ────────────────────────────────────
latest_result = None

def on_result(result, output_image, timestamp_ms):
    global latest_result
    latest_result = result

model_path = os.path.abspath("hand_landmarker.task")

options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
    running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=on_result
)

CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

def get_distance(p1, p2):
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])

def draw_bar(img, x, y, w, h, pct, color, label):
    """Draw a vertical bar indicator"""
    cv2.rectangle(img, (x, y), (x+w, y+h), (50,50,50), -1)
    filled = int(h * pct / 100)
    cv2.rectangle(img, (x, y + h - filled), (x+w, y+h), color, -1)
    cv2.rectangle(img, (x, y), (x+w, y+h), (200,200,200), 2)
    cv2.putText(img, f"{label}: {int(pct)}%", (x - 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

# ── Main Loop ──────────────────────────────────────────
cap = cv2.VideoCapture(0)
timestamp = 0

with mp.tasks.vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        success, img = cap.read()
        if not success:
            break

        img = cv2.flip(img, 1)  # Mirror view
        h, w, _ = img.shape

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        landmarker.detect_async(mp_img, timestamp)
        timestamp += 1

        vol_pct = 0
        bright_pct = 0

        if latest_result and latest_result.hand_landmarks:
            hands_data = latest_result.hand_landmarks
            handedness = latest_result.handedness if latest_result.handedness else []

            for i, hand in enumerate(hands_data):

_IncompleteInputError: incomplete input (2398972841.py, line 83)